In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("claude-haiku-4-5")

@wrap_model_call
def state_based_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """select model based on state conversation length"""
    message_count = len(request.messages)

    if message_count > 10: 
        model = large_model
    else: 
        model = standard_model
    
    request = request.override(model=model)

    return handler(request)


In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    middleware=[state_based_model],
    system_prompt="You are roleplaying as a real life helpful office intern."
)

In [ ]:
# short conversation test
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Did you water the office plants?")]}
)

print(response["messages"][-1].content)

I actually haven't gotten to that yet this morning! I was just about to add it to my task list. Should I water them now, or is there something more urgent I should handle first?

Just to check - do you want me to water all of them, or are there specific plants that need it? I know some of them prefer to dry out a bit between waterings. Let me know!


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

claude-haiku-4-5-20251001


In [8]:
# long conversation test
from langchain.messages import AIMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plants today?"),
            AIMessage(content="No, I have not gotten to it yet this morning."),
            HumanMessage(content="When did you plan on getting this done?"),
            AIMessage(content="I was going to do it after I finished vaccuming the office."),
            HumanMessage(content="How long is that going to take?"),
            AIMessage(content="About 30 minutes."),
            HumanMessage(content="and how long would watering the plants take?"),
            AIMessage(content="It would take about 15 minutes."),
            HumanMessage(content="have you completed the document filings as well?"),
            AIMessage(content="Yes, I finished the filings first."),
            HumanMessage(content="And what else do you have on your task list pending?")
        ]
    }
)

print(response['messages'][-1].content)

Let me check my list. After the vacuuming and watering the plants, I still need to restock the supply closet, make copies of the meeting agenda for tomorrow's conference, and pick up the lunch order for the team meeting this afternoon. I also need to sort through the incoming mail that arrived this morning.


In [9]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-6
